In [11]:
from core.dataset import Dataset as MyDataset
from monai.data import DatasetFunc, ImageDataset
from helpers.paths import DATA_ROOT

In [2]:
my_dataset = MyDataset("roi_train2")

img_list = []
seg_list = []
for subid in my_dataset.subjects:
    subject = my_dataset.subject(subid)
    img_path = subject.dir / "flair.phase.nii.gz"
    seg_path = subject.dir / "prl_seg_def_prob.nii.gz"
    if img_path.exists():
        img_list.append(img_path)
        seg_list.append(seg_path)

img_dataset = ImageDataset(
    image_files=img_list,
    seg_files=seg_list,
    image_only=False
)

In [3]:
import random
from collections import defaultdict
from math import floor

import numpy as np


def stratified_subject_fold_assignment(
    subjects: list[dict],
    n_folds: int = 5,
    test_fraction: float = 0.2,
    seed: int = 42,
) -> dict:
    """Assign subjects to folds with per-subject stratification.

    All lesions from one subject go to the same fold. Subjects are stratified
    by PRL count to distribute PRL cases evenly across folds and testing.

    Strategy:
    1. Bucket subjects by PRL count (0, 1, 2, 3+).
    2. Within each bucket, shuffle and split off test_fraction for testing.
    3. Remaining subjects are assigned to folds round-robin within each bucket.

    Args:
        subjects: List of dicts with subject_id, n_lesions, n_prls.
        n_folds: Number of training folds.
        test_fraction: Fraction of subjects for testing.
        seed: Random seed.

    Returns:
        Dict with:
            training: list of {subject_id, fold, n_lesions, n_prls}
            testing: list of {subject_id, n_lesions, n_prls}
            fold_summary: dict of per-fold statistics
    """
    rng = random.Random(seed)

    # Bucket subjects by PRL count
    buckets: dict[int, list[dict]] = defaultdict(list)
    for subj in subjects:
        bucket_key = min(subj["n_prls"], 3)  # 0, 1, 2, 3+
        buckets[bucket_key].append(subj)

    training = []
    testing = []

    for bucket_key in sorted(buckets.keys()):
        bucket = list(buckets[bucket_key])
        rng.shuffle(bucket)

        n_test = max(1, floor(len(bucket) * test_fraction)) if len(bucket) > 1 else 0
        test_subjects = bucket[:n_test]
        train_subjects = bucket[n_test:]

        for subj in test_subjects:
            testing.append({
                "subject_id": subj["subject_id"],
                "n_lesions": subj["n_lesions"],
                "n_prls": subj["n_prls"],
            })

        for i, subj in enumerate(train_subjects):
            fold = i % n_folds
            training.append({
                "subject_id": subj["subject_id"],
                "fold": fold,
                "n_lesions": subj["n_lesions"],
                "n_prls": subj["n_prls"],
            })

    # Compute summary statistics
    fold_summary = {}
    for fold_num in range(n_folds):
        fold_subjs = [t for t in training if t["fold"] == fold_num]
        fold_summary[fold_num] = {
            "n_subjects": len(fold_subjs),
            "n_lesions": sum(s["n_lesions"] for s in fold_subjs),
            "n_prls": sum(s["n_prls"] for s in fold_subjs),
        }
    test_summary = {
        "n_subjects": len(testing),
        "n_lesions": sum(s["n_lesions"] for s in testing),
        "n_prls": sum(s["n_prls"] for s in testing),
    }
    fold_summary["testing"] = test_summary

    return {
        "training": training,
        "testing": testing,
        "fold_summary": fold_summary,
    }

In [4]:
import pandas as pd
import json
import json
from pathlib import Path

train_home = Path("/home/srs-9/Projects/prl_project/training/full_brain")
labels_to_use = pd.read_csv(train_home/"labels_to_use.csv", index_col="subid")
prl_df = pd.read_csv("/home/srs-9/Projects/prl_project/src/resources/PRL_spreadsheet-lstai_update_label_reference.csv", index_col="subid")

In [13]:
import math

prl_df = prl_df.loc[labels_to_use.index, :]
prl_df = prl_df.sort_values(by="Total PRL", ascending=False)
subjects = prl_df.rename(columns={"Total PRL": "lesions"})
K = 6
T = len(prl_df)

groups = [[] for _ in range(K)]
group_counts = [0] * K
group_sums = [0] * K
target_size =  math.ceil(T/K)  # assume divisible for simplicity

# greedy initialization
for subid, s in subjects.iterrows():
    eligible = [k for k in range(K) if group_counts[k] < target_size]
    k_best = min(eligible, key=lambda k: group_sums[k])
    groups[k_best].append(subid)
    group_counts[k_best] += 1
    group_sums[k_best] += s.lesions

In [17]:
group_sums

[10, 10, 10, 9, 8, 8]

In [31]:
dataroot = DATA_ROOT
training = []
f = 0
for group in groups[:-1]:
    for subid in group:
        subject_root = dataroot / f"sub{subid}-{prl_df.loc[subid, 'date_mri']}"
        image = subject_root / "flair.phase.nii.gz"
        assert image.exists()
        seg = subject_root / "prl_seg_def_prob.nii.gz"
        assert seg.exists()
        struct = {
            "subid": subid,
            "nPRL": int(prl_df.loc[subid, "Total PRL"]),
            "image": str(image.relative_to(dataroot)),
            "label": str(seg.relative_to(dataroot)),
            "fold": f
        }
        training.append(struct)
    f += 1

testing = []
for subid in groups[-1]:
    subject_root = dataroot / f"sub{subid}-{prl_df.loc[subid, 'date_mri']}"
    image = subject_root / "flair.phase.nii.gz"
    assert image.exists()
    seg = subject_root / "prl_seg_def_prob.nii.gz"
    assert seg.exists()
    struct = {
        "subid": subid,
        "nPRL": int(prl_df.loc[subid, "Total PRL"]),
        "image": str(image.relative_to(dataroot)),
        "label": str(seg.relative_to(dataroot)),
    }
    testing.append(struct)

In [33]:
datalist = {
    "dataroot": str(dataroot),
    "training": training,
    "testing": testing
}
with open("/home/srs-9/Projects/prl_project/training/full_brain/datalist.json", 'w') as f:
    json.dump(datalist, f, indent=4)

In [ ]:
 # swap refinement
# improved = True
# while improved:
#     improved = False
#     for a in range(K):
#         for b in range(a+1, K):
#             for i, s1 in enumerate(groups[a]):
#                 for j, s2 in enumerate(groups[b]):
#                     old_obj = (group_sums[a]-target)**2 + (group_sums[b]-target)**2
#                     new_sum_a = group_sums[a] - s1.lesions + s2.lesions
#                     new_sum_b = group_sums[b] - s2.lesions + s1.lesions
#                     new_obj = (new_sum_a-target)**2 + (new_sum_b-target)**2

#                     if new_obj < old_obj:
#                         groups[a][i], groups[b][j] = groups[b][j], groups[a][i]
#                         group_sums[a] = new_sum_a
#                         group_sums[b] = new_sum_b
#                         improved = True

[]